# Credit Card: Hybrid Resampling Methods
Thực nghiệm kết hợp Over-sampling và Under-sampling:
1. **SMOTE + Tomek Links (`SMOTETomek`)**: Sinh mẫu thiểu số nhân tạo rồi dọn sạch các cặp điểm đối lập ở biên phân chia.
2. **SMOTE + Edited Nearest Neighbours (`SMOTEENN`)**: Loại bỏ các mẫu bị phân lớp sai bởi k-NN sau khi tăng cường mẫu.
3. Đánh giá trên Logistic Regression và LightGBM.


In [ ]:
import pandas as pd
from ipynb.fs.defs.preprocessing import get_data
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from imblearn.combine import SMOTETomek, SMOTEENN
import lightgbm as lgb
from util import summary


In [ ]:
X_train, X_test, y_train, y_test = get_data(use_val=False, test_size=0.3)
print("X_train shape:", X_train.shape, "Fraud count:", int(y_train.sum()))


### 1. SMOTE + Tomek Links với Logistic Regression

In [ ]:
smt = SMOTETomek(random_state=42)
X_res, y_res = smt.fit_resample(X_train, y_train)
print("Sau SMOTE-Tomek:", X_res.shape, "Fraud count:", int(y_res.sum()))

clf_lr = Pipeline([("scaler", StandardScaler()), ("logreg", LogisticRegression(max_iter=1000))])
clf_lr.fit(X_res, y_res)
y_pred_lr = clf_lr.predict(X_test)
summary(y_test, y_pred_lr)


### 2. SMOTE + Tomek Links với LightGBM

In [ ]:
lgb_clf = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1)
lgb_clf.fit(X_res, y_res)
y_pred_lgb = lgb_clf.predict(X_test)
summary(y_test, y_pred_lgb)


### 3. SMOTE + ENN với LightGBM

In [ ]:
sme = SMOTEENN(random_state=42)
X_res_enn, y_res_enn = sme.fit_resample(X_train, y_train)
print("Sau SMOTE-ENN:", X_res_enn.shape, "Fraud count:", int(y_res_enn.sum()))

lgb_enn = lgb.LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=31, random_state=42, verbose=-1)
lgb_enn.fit(X_res_enn, y_res_enn)
y_pred_enn = lgb_enn.predict(X_test)
summary(y_test, y_pred_enn)
